In [ ]:
from google.colab import drive
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')

# --- SETTINGS ---
BASE_PATH = '/content/drive/MyDrive/nutrient-deficiency'
IMG_SIZE = 224
BATCH_SIZE = 32

crop_map = {'rice_deficiency': 0, 'maize_deficiency': 1, 'coffee_deficiency': 2}
label_map = {'Nitrogen(N)': 0, 'Phosphorus(P)': 1, 'Potassium(K)': 2, 'Healthy': 3}

Mounted at /content/drive


In [ ]:
# Create the Master List
data = []
for crop_folder, crop_id in crop_map.items():
    crop_path = os.path.join(BASE_PATH, crop_folder)
    for label_name, label_id in label_map.items():
        f_path = os.path.join(crop_path, label_name)
        if os.path.exists(f_path):
            for img in os.listdir(f_path):
                if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                    data.append([os.path.join(f_path, img), crop_id, label_id])

df = pd.DataFrame(data, columns=['filepath', 'crop_id', 'label_id'])
df.head()

,filepath,crop_id,label_id
0,/content/drive/MyDrive/nutrient-deficiency/ric...,0,0
1,/content/drive/MyDrive/nutrient-deficiency/ric...,0,0
2,/content/drive/MyDrive/nutrient-deficiency/ric...,0,0
3,/content/drive/MyDrive/nutrient-deficiency/ric...,0,0
4,/content/drive/MyDrive/nutrient-deficiency/ric...,0,0


In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label_id'], random_state=42)

In [ ]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = img / 255.0
    return img

In [ ]:
# Training Dataset Pipeline
train_ds = tf.data.Dataset.from_tensor_slices((train_df['filepath'], train_df['crop_id'], train_df['label_id']))
train_ds = train_ds.map(lambda f, c, l: (
    (load_image(f), tf.one_hot(c, 3)),
    tf.one_hot(l, 4)
), num_parallel_calls=tf.data.AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Validation Dataset Pipeline
val_ds = tf.data.Dataset.from_tensor_slices((val_df['filepath'], val_df['crop_id'], val_df['label_id']))
val_ds = val_ds.map(lambda f, c, l: (
    (load_image(f), tf.one_hot(c, 3)),
    tf.one_hot(l, 4)
)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# --- BUILD MODEL ---
img_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")
crop_in = layers.Input(shape=(3,), name="crop_input")

# Image Backbone
base = tf.keras.applications.EfficientNetV2S(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False

x = layers.GlobalAveragePooling2D()(base(img_in))
x = layers.Dropout(0.3)(x)

# Fusion
y = layers.Dense(16, activation='relu')(crop_in)
merged = layers.Concatenate()([x, y])
z = layers.Dense(256, activation='relu')(merged)
z = layers.Dropout(0.4)(z)
out = layers.Dense(4, activation='softmax')(z)

model = models.Model(inputs=[img_in, crop_in], outputs=out)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models

# --- 1. IMAGE INPUT BRANCH ---
img_input = layers.Input(shape=(224, 224, 3), name="image_input")

# Load EfficientNetV2-S without the top classification layers
base_model = tf.keras.applications.EfficientNetV2S(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Keep the pre-trained weights frozen for now

# Process image features
x = base_model(img_input)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)

# --- 2. CROP ID INPUT BRANCH ---
# Input for Crop Type: Rice, Maize, Coffee (One-hot encoded, length 3)
crop_input = layers.Input(shape=(3,), name="crop_input")
y = layers.Dense(16, activation='relu')(crop_input)

# --- 3. THE FUSION ---
# Merging the visual features with the crop metadata
combined = layers.Concatenate()([x, y])

# Final Classification Head
z = layers.Dense(128, activation='relu')(combined)
z = layers.Dropout(0.3)(z)

# Output: 3 Classes (Stage 1, 2, or 3)
output = layers.Dense(3, activation='softmax', name="stage_output")(z)

# --- 4. ASSEMBLE & COMPILE ---
growth_model = models.Model(inputs=[img_input, crop_input], outputs=output)

growth_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Print the architecture to verify the connections
growth_model.summary()

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetv2-s    │ (None, 7, 7,      │ 20,331,360 │ image_input[0][0] │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ efficientnetv2-s… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    327,936 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ crop_input          │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │         64 │ crop_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 272)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     34,944 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage_output        │ (None, 3)         │        387 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,694,691 (78.94 MB)

 Trainable params: 363,331 (1.39 MB)

 Non-trainable params: 20,331,360 (77.56 MB)

In [5]:
import tensorflow as tf

# Path where your model is saved
model_path = '/content/drive/MyDrive/nutrient-deficiency/best_growth_fusion_model.keras'

# Load the model
growth_model = tf.keras.models.load_model(model_path)

# UNFREEZE: Allow the pre-trained layers to adjust to your specific data
growth_model.trainable = True

# Re-compile with a MUCH smaller learning rate (1e-5)
# Using a high rate like 1e-3 now would ruin the progress you've made
growth_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model loaded and unfrozen for fine-tuning.")

✅ Model loaded and unfrozen for fine-tuning.


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

# New save path for the improved version
refined_checkpoint = '/content/drive/MyDrive/nutrient-deficiency/best_growth_fusion_model_v2.keras'

callbacks = [
    ModelCheckpoint(refined_checkpoint, save_best_only=True, monitor='val_accuracy', mode='max'),
    # If val_loss doesn't improve for 2 epochs, drop the learning rate further
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7),
    # Stop if it starts overfitting (val_loss going up while train_loss goes down)
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
]

print("🚀 Continuing training to push past 82.64%...")
history = growth_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)

🚀 Continuing training to push past 82.64%...
Epoch 1/30
224/224 ━━━━━━━━━━━━━━━━━━━━ 3209s 14s/step - accuracy: 0.7709 - loss: 0.5424 - val_accuracy: 0.8252 - val_loss: 0.4202 - learning_rate: 1.0000e-05
Epoch 2/30
224/224 ━━━━━━━━━━━━━━━━━━━━ 137s 611ms/step - accuracy: 0.7708 - loss: 0.5333 - val_accuracy: 0.8264 - val_loss: 0.4194 - learning_rate: 1.0000e-05
Epoch 3/30
224/224 ━━━━━━━━━━━━━━━━━━━━ 136s 606ms/step - accuracy: 0.7687 - loss: 0.5353 - val_accuracy: 0.8264 - val_loss: 0.4200 - learning_rate: 1.0000e-05
Epoch 4/30
224/224 ━━━━━━━━━━━━━━━━━━━━ 121s 514ms/step - accuracy: 0.7734 - loss: 0.5277 - val_accuracy: 0.8247 - val_loss: 0.4209 - learning_rate: 1.0000e-05
Epoch 5/30
224/224 ━━━━━━━━━━━━━━━━━━━━ 141s 509ms/step - accuracy: 0.7720 - loss: 0.5320 - val_accuracy: 0.8252 - val_loss: 0.4211 - learning_rate: 2.0000e-06
Epoch 6/30
224/224 ━━━━━━━━━━━━━━━━━━━━ 131s 587ms/step - accuracy: 0.7715 - loss: 0.5292 - val_accuracy: 0.8230 - val_loss: 0.4209 - learning_rate: 2.0000e

#run from here


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# --- CONFIGURATION ---

BASE_PATH = '/content/drive/MyDrive/nutrient-deficiency'
CROP_FOLDERS = {
    'rice_growth_stage': 0,
    'maize_growth_stage': 1,
    'coffee_growth_stage': 2
}
# We map your folder names (1, 2, 3) to 0-indexed labels (0, 1, 2)
STAGE_MAP = {'1': 0, '2': 1, '3': 2}

# --- DATA COLLECTION ---
all_data = []
for crop_name, crop_id in CROP_FOLDERS.items():
    crop_dir = os.path.join(BASE_PATH, crop_name)
    for stage_folder, stage_id in STAGE_MAP.items():
        folder_path = os.path.join(crop_dir, stage_folder)
        if os.path.exists(folder_path):
            images = [os.path.join(folder_path, f) for f in os.listdir(folder_path)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            for img_path in images:
                all_data.append({'filepath': img_path, 'crop_id': crop_id, 'stage_id': stage_id})

df = pd.DataFrame(all_data)
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df[['crop_id', 'stage_id']], random_state=42)

print(f"✅ Dataset Loaded: {len(df)} total images.")

✅ Dataset Loaded: 8954 total images.


In [2]:
import tensorflow as tf

def load_and_preprocess(filepath, crop_id, stage_id):
    # 1. Load and process Image
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224]) / 255.0

    # 2. Convert IDs to One-Hot (3 classes each)
    crop_oh = tf.one_hot(crop_id, 3)
    stage_oh = tf.one_hot(stage_id, 3)

    # Return: ( [Image, CropID], StageID )
    return (img, crop_oh), stage_oh

# Create the Training Dataset
train_gen = tf.data.Dataset.from_tensor_slices((
    train_df['filepath'].values,
    train_df['crop_id'].values,
    train_df['stage_id'].values
))

train_gen = train_gen.shuffle(len(train_df)).map(load_and_preprocess).batch(32).prefetch(tf.data.AUTOTUNE)

# Create the Validation Dataset
val_gen = tf.data.Dataset.from_tensor_slices((
    val_df['filepath'].values,
    val_df['crop_id'].values,
    val_df['stage_id'].values
))

val_gen = val_gen.map(load_and_preprocess).batch(32).prefetch(tf.data.AUTOTUNE)

print("✅ Data pipeline is ready using tf.data!")

✅ Data pipeline is ready using tf.data!


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
import pandas as pd

IMG_SIZE = 300
BATCH_SIZE = 20 # Lowered batch size to 16 to avoid 'Out of Memory' at 300px

def load_and_preprocess_300(filepath, crop_id, stage_id, augment=True):
    # Image processing
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.8, 1.2)

    img = img / 255.0

    # Dual inputs and labels
    crop_oh = tf.one_hot(crop_id, 3)
    stage_oh = tf.one_hot(stage_id, 3)

    return (img, crop_oh), stage_oh

# Create Datasets
train_gen = tf.data.Dataset.from_tensor_slices((
    train_df['filepath'].values, train_df['crop_id'].values, train_df['stage_id'].values
)).shuffle(len(train_df)).map(lambda f, c, s: load_and_preprocess_300(f, c, s, augment=True)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_gen = tf.data.Dataset.from_tensor_slices((
    val_df['filepath'].values, val_df['crop_id'].values, val_df['stage_id'].values
)).map(lambda f, c, s: load_and_preprocess_300(f, c, s, augment=False)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. LOAD YOUR EXISTING 82% MODEL
old_model_path = '/content/drive/MyDrive/nutrient-deficiency/best_growth_fusion_model.keras'
old_model = tf.keras.models.load_model(old_model_path) #

# 2. CREATE THE NEW 300px SHELL
img_input = layers.Input(shape=(300, 300, 3), name="image_input")
crop_input = layers.Input(shape=(3,), name="crop_input")

# Use the SAME backbone as before (EfficientNetV2S)
base_model_300 = tf.keras.applications.EfficientNetV2S(
    input_tensor=img_input,
    include_top=False,
    weights=None # We will fill this in a second
)

# Rebuild the Head exactly as shown in your image_b77dc4.png
x = layers.GlobalAveragePooling2D()(base_model_300.output)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)

y = layers.Dense(16, activation='relu')(crop_input)

combined = layers.Concatenate()([x, y])
z = layers.Dense(128, activation='relu')(combined)
z = layers.Dropout(0.3)(z)
output = layers.Dense(3, activation='softmax', name="stage_output")(z)

growth_model_300 = models.Model(inputs=[img_input, crop_input], outputs=output)

# 3. TRANSFER KNOWLEDGE (The Fix)
# Instead of by_name=True, we manually set the weights from the old layers
for i in range(len(old_model.layers)):
    try:
        growth_model_300.layers[i].set_weights(old_model.layers[i].get_weights())
    except:
        # This skips the Input layer which changed from 224 to 300
        continue

growth_model_300.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Knowledge transferred successfully without legacy errors!")

✅ Knowledge transferred successfully without legacy errors!


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

# This is your safety file on Google Drive
checkpoint_path = '/content/drive/MyDrive/nutrient-deficiency/growth_300px_autosave.keras'

callbacks = [
    # SAFETY: This saves the model immediately after any epoch where accuracy improves
    ModelCheckpoint(
        filepath=checkpoint_path,
        save_best_only=True,
        monitor='val_accuracy',
        mode='max',
        verbose=1 # This will print "Epoch 001: val_accuracy improved from... to..."
    ),

    # Existing logic to handle plateaus
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=1),

    # Stops if the model starts "crashing" in quality/overfitting
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]

print(" Starting training with auto-save enabled.")
history = growth_model_300.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=callbacks
)

🚀 Starting training with auto-save enabled.
Epoch 1/20
